# DataCentric-Env — GRPO Training Notebook

Trains Qwen2.5-3B-Instruct as a data quality agent using GRPO.

**Sections:**
1. Install dependencies
2. Model setup (Qwen2.5-3B-Instruct, 4-bit LoRA)
3. Rollout function
4. Collect training data
5. GRPO training loop
6. Save model via Unsloth merge path

In [ ]:
# Cell 1: Install dependencies
!pip install unsloth trl transformers accelerate peft datasets requests

In [ ]:
# Cell 2: Imports and config
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
import requests, json, torch

ENV_URL = 'https://your-hf-username-datacentric-env.hf.space'  # set your HF Space URL

SYSTEM_PROMPT = (
    'You are a data quality agent. You receive dataset statistics and must choose '
    'which specialist tool to call to improve the dataset so a downstream classifier '
    'performs better.\n\n'
    'Always respond with valid JSON in this exact format:\n'
    '{"agent": "<tool_name>", "target": "<column_or_all>", "strategy": "<strategy_name>"}\n\n'
    'Available tools: cleaner, augmenter, balancer, relabeler, validator\n'
    'Cleaner strategies: median_impute, mean_impute, drop_rows\n'
    'Balancer strategies: undersample\n'
    'Relabeler: use when labels are noisy, costs 2 budget points.'
)
print('Imports OK')

In [ ]:
# Cell 3: Model setup
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32)
print('Model loaded')

In [ ]:
# Cell 4: Rollout function
def build_prompt(obs):
    return SYSTEM_PROMPT + '\n\nCurrent state:\n' + json.dumps(obs, indent=2) + '\n\nYour action:'

def rollout(prompt='start'):
    obs = requests.post(ENV_URL + '/reset').json()
    trajectories = []
    for step in range(10):
        full_prompt = build_prompt(obs)
        inputs = tokenizer(full_prompt, return_tensors='pt').to('cuda')
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7)
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        try:
            action = json.loads(response.strip())
        except Exception:
            action = {'agent': 'validator'}
        result = requests.post(ENV_URL + '/step', json=action).json()
        reward = result.get('reward', -1.0)
        trajectories.append({'prompt': full_prompt, 'response': response, 'reward': reward})
        obs = result.get('observation', obs)
        if result.get('done'):
            break
    return trajectories

print('Rollout function defined')

In [ ]:
# Cell 5: Collect rollouts and build dataset
print('Collecting rollouts...')
all_trajectories = []
for episode in range(50):
    all_trajectories.extend(rollout('start'))
    if episode % 10 == 0:
        print(f'  Episode {episode}/50 collected')

dataset = Dataset.from_list([
    {'prompt': t['prompt'], 'chosen': t['response'], 'reward': t['reward']}
    for t in all_trajectories
])
print(f'Dataset size: {len(dataset)}')

In [ ]:
# Cell 6: GRPO training
config = GRPOConfig(
    output_dir='./datacentric-grpo',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=100,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()

In [ ]:
# Cell 7: Save via Unsloth merge path
# IMPORTANT: do NOT use naive save_pretrained — use Unsloth merge path
model.save_pretrained_merged(
    'datacentric-grpo-final',
    tokenizer,
    save_method='merged_16bit',
)
print('Training complete. Model saved to datacentric-grpo-final/')